# Paradigm 2 — Retrieval-Augmented Recommenders

**Anchor paper**: K-RagRec (Wang et al., ACL 2025) — Knowledge Graph RAG for LLM-based Recommendation

**Architecture**: Dense retrieval for candidate generation + LLM reranking/explanation.
User history is encoded via sentence embeddings, similar items are retrieved from an
embedding index, and an LLM reranks candidates and generates explanations using
the retrieved context.

**Key insight from K-RagRec**: External knowledge (knowledge graphs, item metadata)
retrieved at inference time grounds the LLM's recommendations, reducing hallucination
and improving recommendation quality over pure generative approaches.

**Our implementation**: Sentence-transformers for item embeddings, FAISS for retrieval,
LLM for reranking and explanation. Both API and local LLaMA options.

**Ranking metrics**: HR@K, NDCG@K (from LLM reranking)  
**Explanation metrics**: Relevance, Specificity, Hallucination rate

In [ ]:
!pip install -q --upgrade sentence-transformers faiss-cpu numpy pandas huggingface_hub

In [ ]:
import sys
sys.path.insert(0, '.')

import json
import pickle
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from tutorial_utils import (
    evaluate_ranking, evaluate_explanations,
    save_results, save_predictions, LatencyTracker,
    DATA_DIR, RESULTS_DIR
)

with open(DATA_DIR / 'shared_data.pkl', 'rb') as f:
    shared = pickle.load(f)

user_histories = shared['user_histories']
test_ground_truth = shared['test_ground_truth']
item_titles = shared['item_titles']
all_items = shared['all_items']
n_users = shared['n_users']
n_items = shared['n_items']

print(f"Users: {n_users}, Items: {n_items}, Test: {len(test_ground_truth)}")

## 1. Build item embedding index

In [ ]:
# Encode all item titles with sentence-transformers
encoder = SentenceTransformer('all-MiniLM-L6-v2')  # fast, 384-dim

# Build ordered list of (item_idx, title)
item_indices = sorted(item_titles.keys())
titles_ordered = [item_titles[idx] for idx in item_indices]
idx_to_pos = {idx: pos for pos, idx in enumerate(item_indices)}
pos_to_idx = {pos: idx for pos, idx in enumerate(item_indices)}

print(f"Encoding {len(titles_ordered)} item titles...")
item_embeddings = encoder.encode(titles_ordered, show_progress_bar=True, batch_size=256)
item_embeddings = item_embeddings.astype('float32')

# Normalize for cosine similarity
faiss.normalize_L2(item_embeddings)

# Build FAISS index
dim = item_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # inner product on normalized vectors = cosine similarity
index.add(item_embeddings)
print(f"FAISS index built: {index.ntotal} items, {dim}-dim")

## 2. Retrieval: find candidates from user history

In [ ]:
def retrieve_candidates(user_idx, top_k=50, n_history=10):
    """Retrieve candidate items by encoding user's recent history and querying FAISS."""
    history = user_histories.get(user_idx, [])
    if not history:
        return []

    # Encode recent history titles as a single query
    recent = history[-n_history:]
    recent_titles = [item_titles.get(it, '') for it in recent if it in item_titles]
    if not recent_titles:
        return []

    query_text = "Books similar to: " + "; ".join(recent_titles)
    query_emb = encoder.encode([query_text]).astype('float32')
    faiss.normalize_L2(query_emb)

    # Retrieve more than needed, then filter out history
    scores, positions = index.search(query_emb, top_k + len(history))
    history_set = set(history)

    candidates = []
    for pos, score in zip(positions[0], scores[0]):
        item_idx = pos_to_idx.get(int(pos))
        if item_idx is not None and item_idx not in history_set:
            candidates.append((item_idx, float(score)))
        if len(candidates) >= top_k:
            break

    return candidates

In [ ]:
# Test retrieval for one user
sample_user = list(test_ground_truth.keys())[0]
cands = retrieve_candidates(sample_user, top_k=10)
true_item = test_ground_truth[sample_user]

print(f"User {sample_user} history (last 5):")
for it in user_histories[sample_user][-5:]:
    print(f"  - {item_titles.get(it, '?')}")
print(f"\nTrue next item: {item_titles.get(true_item, '?')}")
print(f"\nRetrieved candidates:")
for item_idx, score in cands[:5]:
    marker = " <-- HIT" if item_idx == true_item else ""
    print(f"  {score:.3f}  {item_titles.get(item_idx, '?')}{marker}")

## 3. LLM Reranking

The retrieval step gives us a candidate set. Now we use an LLM to rerank
these candidates based on the user's full reading context.

In [ ]:
RERANK_PROMPT = """You are a book recommendation system. A user has read these books:
{history}

Here are candidate books to recommend (numbered):
{candidates}

Rank the top 10 candidates from most to least relevant for this user.
Return ONLY a JSON list of the candidate numbers in order, e.g. [3, 1, 7, ...].
Then on a new line, write a 2-3 sentence explanation for your top recommendation."""


def format_rerank_prompt(user_idx, candidates):
    history = user_histories.get(user_idx, [])
    history_titles = [item_titles.get(it, '?') for it in history[-10:]]
    cand_lines = []
    for i, (item_idx, score) in enumerate(candidates[:20]):
        cand_lines.append(f"{i+1}. {item_titles.get(item_idx, '?')}")
    return RERANK_PROMPT.format(
        history='\n'.join(f'- {t}' for t in history_titles),
        candidates='\n'.join(cand_lines),
    )

In [ ]:
# --- Option A: API-based reranking ---
# import openai
# client = openai.OpenAI(api_key="YOUR_KEY")
#
# def rerank_api(user_idx, candidates):
#     prompt = format_rerank_prompt(user_idx, candidates)
#     response = client.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[{"role": "user", "content": prompt}],
#         max_tokens=300,
#         temperature=0.1,
#     )
#     text = response.choices[0].message.content.strip()
#     # Parse ranked indices from JSON list
#     try:
#         import re
#         match = re.search(r'\[([\d,\s]+)\]', text)
#         if match:
#             ranked_nums = [int(x.strip()) for x in match.group(1).split(',')]
#             ranked_items = [candidates[n-1][0] for n in ranked_nums if 1 <= n <= len(candidates)]
#         else:
#             ranked_items = [c[0] for c in candidates]  # fallback
#     except Exception:
#         ranked_items = [c[0] for c in candidates]
#     # Extract explanation (text after the JSON list)
#     explanation = text.split(']', 1)[-1].strip() if ']' in text else ""
#     return ranked_items, explanation

In [ ]:
# --- Retrieval-only baseline (no LLM reranking) ---
# Use this when LLM is not available; uses retrieval scores directly.

def rerank_retrieval_only(user_idx, candidates):
    """No reranking — just use retrieval similarity scores."""
    ranked_items = [c[0] for c in candidates]
    if candidates:
        top_title = item_titles.get(candidates[0][0], '?')
        history = user_histories.get(user_idx, [])
        last_title = item_titles.get(history[-1], '?') if history else '?'
        explanation = (f"Based on your recent reading of '{last_title}', "
                      f"we recommend '{top_title}' as it covers similar topics.")
    else:
        explanation = ""
    return ranked_items, explanation

## 4. Full RAG pipeline: retrieve + rerank

In [ ]:
tracker = LatencyTracker()
predictions = {}
explanations = {}

# Choose your reranker: rerank_api or rerank_retrieval_only
rerank_fn = rerank_retrieval_only

for i, user_idx in enumerate(test_ground_truth):
    tracker.start()
    candidates = retrieve_candidates(user_idx, top_k=50)
    ranked_items, explanation = rerank_fn(user_idx, candidates)
    predictions[user_idx] = ranked_items[:20]
    explanations[user_idx] = {"item_idx": ranked_items[0] if ranked_items else -1, "text": explanation}
    tracker.stop()

    if (i + 1) % 1000 == 0:
        print(f"  Processed {i+1}/{len(test_ground_truth)} users")

print(f"Done. {len(predictions)} predictions generated.")

## 5. Evaluate

In [ ]:
ranking_results = evaluate_ranking(predictions, test_ground_truth, k_values=[5, 10, 20])
explanation_results = evaluate_explanations(
    {u: explanations[u] for u in list(explanations.keys())[:100]},
    user_histories, item_titles, set(item_titles.values())
)
latency = tracker.summary()

print("Ranking results:")
for metric, val in ranking_results.items():
    print(f"  {metric}: {val:.4f}")

print("\nExplanation quality:")
for metric, val in explanation_results.items():
    print(f"  {metric}: {val:.4f}")

print(f"\nLatency: {latency['mean_latency_ms']:.2f} ms/user")

## 6. Save results

In [ ]:
results = {
    "paradigm": "rag",
    "model": "Sentence-Transformers retrieval + LLM rerank",
    "anchor_paper": "K-RagRec (Wang et al., ACL 2025)",
    "ranking": ranking_results,
    "explanation": explanation_results,
    "system": {"latency": latency},
}

save_results("rag", results)
save_predictions("rag", predictions)
print("Done.")